# MIMII Fan — Anomaly Detection (General-Use)

Robust pipeline that works across machine IDs and SNR levels without per-machine tuning:

- **Features**: log-mel stats + MFCC stats, extracted after bandpass filtering (helps low-SNR recordings)
- **Models**: Autoencoder (primary, 85% weight) + GMM (secondary, 15% weight)
- **Threshold**: 99th-percentile of training-normal scores — robust when val abnormal count is small
- **No per-machine hyperparameter tuning required**


## 1. Imports & Config

In [6]:
import warnings
warnings.filterwarnings('ignore')

import re
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.feature
import librosa.effects
from scipy.signal import butter, sosfilt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_DIR     = Path(r"D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan")
CACHE_FILE      = Path(".cache_fan_v2.npz")
FORCE_RECOMPUTE = False   # set True to recompute features from scratch

# ── Audio ──────────────────────────────────────────────────────────────────────
SR          = 16_000
DURATION    = 10.0
N_FFT       = 1024
HOP_LENGTH  = 512
N_MELS      = 64
N_MFCC      = 40
FMIN        = 50.0
FMAX        = 8_000.0

# ── Bandpass filter ────────────────────────────────────────────────────────────
# Removes noise floor below 300 Hz and high-freq hiss above 4 kHz.
# Applied to ALL recordings — doesn't hurt clean ones, significantly helps -6 dB.
BP_LOW_HZ  = 300
BP_HIGH_HZ = 4_000
BP_ORDER   = 4

# ── GMM ────────────────────────────────────────────────────────────────────────
GMM_COMPONENTS = 8
GMM_COV        = 'full'
GMM_REG        = 1e-4

# ── Autoencoder ────────────────────────────────────────────────────────────────
AE_LATENT   = 32    # bumped from 16 — better representation for 212-dim input
AE_EPOCHS   = 120
AE_LR       = 1e-3
AE_BATCH    = 64
AE_PATIENCE = 15

# ── Ensemble weights: AE dominant ─────────────────────────────────────────────
# AE consistently outperformed GMM (0.87 vs 0.79 mean AUC).
# GMM kept at low weight as a complementary signal only.
W_AE  = 0.85
W_GMM = 0.15

# ── Threshold strategy ─────────────────────────────────────────────────────────
# 99th percentile of training-normal scores — fully unsupervised, no abnormal
# labels needed. Youden-J on val is unreliable when val has few abnormals.
THRESHOLD_QUANTILE = 0.99

SEED   = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {DEVICE}")
print("Config ready.")

Device : cuda
Config ready.


## 2. Feature Extraction

Each 10 s clip → one 212-dim vector:
- **Bandpass 300–4000 Hz** before everything else
- mean + std of each log-mel band → 128 values
- mean + std of each MFCC → 80 values
- log-RMS stats (mean, std, min, max) → 4 values

In [7]:
def make_bandpass_sos(low_hz, high_hz, sr, order=4):
    """Butterworth bandpass as second-order sections (numerically stable)."""
    nyq = sr / 2.0
    return butter(order, [low_hz / nyq, high_hz / nyq], btype='band', output='sos')

_BP_SOS = make_bandpass_sos(BP_LOW_HZ, BP_HIGH_HZ, SR, BP_ORDER)


def extract_features(path: str) -> np.ndarray:
    """Load audio, bandpass filter, then extract a compact stat-vector."""
    y, _ = librosa.load(str(path), sr=SR, mono=True, duration=DURATION)

    # Bandpass — suppress noise floor and high-freq hiss
    y = sosfilt(_BP_SOS, y).astype(np.float32)

    # Pre-emphasis after bandpass to sharpen transients
    y = librosa.effects.preemphasis(y, coef=0.97)

    # Peak normalise
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak

    # Log-mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)  # (N_MELS, T)

    # MFCCs
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH
    )  # (N_MFCC, T)

    # Log-RMS energy
    rms     = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    log_rms = np.log(rms + 1e-9)

    feat = np.concatenate([
        log_mel.mean(axis=1),  # N_MELS
        log_mel.std(axis=1),   # N_MELS
        mfcc.mean(axis=1),     # N_MFCC
        mfcc.std(axis=1),      # N_MFCC
        [log_rms.mean(), log_rms.std(), log_rms.min(), log_rms.max()],  # 4
    ])
    return feat.astype(np.float32)


def build_file_index(root: Path) -> pd.DataFrame:
    records = []
    for wav in sorted(root.rglob("*.wav")):
        parts = wav.relative_to(root).parts
        if len(parts) < 3:
            continue
        id_snr, condition = parts[0], parts[1]
        m = re.match(r"id_(\w+)_(.+)", id_snr)
        machine_id = m.group(1) if m else id_snr
        snr        = m.group(2) if m else "?"
        records.append(dict(
            path=str(wav),
            machine_id=machine_id,
            snr=snr,
            label=1 if condition.lower() in {"abnormal", "anomaly"} else 0,
        ))
    return pd.DataFrame(records)


def load_or_compute_features(root: Path, cache: Path, force: bool = False):
    if cache.exists() and not force:
        print(f"Loading cached features from {cache}")
        d = np.load(cache, allow_pickle=True)
        return pd.DataFrame(d['meta'].item()), d['X'], d['y']

    df = build_file_index(root)
    print(f"Found {len(df)} files  ({(df.label==0).sum()} normal, {(df.label==1).sum()} abnormal)")

    feat_dim = 2 * N_MELS + 2 * N_MFCC + 4
    X = np.zeros((len(df), feat_dim), dtype=np.float32)
    for i, row in df.reset_index(drop=True).iterrows():
        try:
            X[i] = extract_features(row['path'])
        except Exception as e:
            print(f"  Skipping {row['path']}: {e}")
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(df)} processed...")

    y = df['label'].values.astype(np.int8)
    np.savez_compressed(str(cache), X=X, y=y, meta=df.to_dict())
    print(f"Saved cache -> {cache}")
    return df, X, y


df, X, y = load_or_compute_features(DATASET_DIR, CACHE_FILE, force=FORCE_RECOMPUTE)
FEAT_DIM  = X.shape[1]
print(f"\nFeature matrix : {X.shape}  ({FEAT_DIM} features per file)")
df.head()

Found 6925 files  (5091 normal, 1834 abnormal)
  200/6925 processed...
  400/6925 processed...
  600/6925 processed...
  800/6925 processed...
  1000/6925 processed...
  1200/6925 processed...
  1400/6925 processed...
  1600/6925 processed...
  1800/6925 processed...
  2000/6925 processed...
  2200/6925 processed...
  2400/6925 processed...
  2600/6925 processed...
  2800/6925 processed...
  3000/6925 processed...
  3200/6925 processed...
  3400/6925 processed...
  3600/6925 processed...
  3800/6925 processed...
  4000/6925 processed...
  4200/6925 processed...
  4400/6925 processed...
  4600/6925 processed...
  4800/6925 processed...
  5000/6925 processed...
  5200/6925 processed...
  5400/6925 processed...
  5600/6925 processed...
  5800/6925 processed...
  6000/6925 processed...
  6200/6925 processed...
  6400/6925 processed...
  6600/6925 processed...
  6800/6925 processed...
Saved cache -> .cache_fan_v2.npz

Feature matrix : (6925, 212)  (212 features per file)


,path,machine_id,snr,label
0,D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...,00,0db,1
1,D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...,00,0db,1
2,D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...,00,0db,1
3,D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...,00,0db,1
4,D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...,00,0db,1


## 3. Models

In [8]:
# ── GMM detector ───────────────────────────────────────────────────────────────
class GMMDetector:
    """Train on normal-only. Score = negative log-likelihood (higher = more anomalous)."""

    def __init__(self):
        self.scaler = StandardScaler()
        self.gmm    = GaussianMixture(
            n_components=GMM_COMPONENTS, covariance_type=GMM_COV,
            reg_covar=GMM_REG, max_iter=400, random_state=SEED
        )

    def fit(self, X_normal: np.ndarray):
        Xn = self.scaler.fit_transform(X_normal)
        self.gmm.fit(Xn)
        print(f"  GMM fitted  BIC={self.gmm.bic(Xn):.0f}  components={GMM_COMPONENTS}")
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        return -self.gmm.score_samples(self.scaler.transform(X))


# ── Autoencoder ────────────────────────────────────────────────────────────────
class AE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = AE_LATENT):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.GELU(),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.GELU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64),  nn.GELU(),
            nn.Linear(64, 128),        nn.BatchNorm1d(128), nn.GELU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


class AEDetector:
    """Train on normal-only. Score = reconstruction error (higher = more anomalous)."""

    def __init__(self, input_dim: int):
        self.scaler = StandardScaler()
        self.model  = AE(input_dim, AE_LATENT).to(DEVICE)

    def fit(self, X_normal: np.ndarray):
        Xn     = torch.tensor(self.scaler.fit_transform(X_normal), dtype=torch.float32)
        loader = DataLoader(TensorDataset(Xn), batch_size=AE_BATCH, shuffle=True)
        opt    = torch.optim.Adam(self.model.parameters(), lr=AE_LR, weight_decay=1e-5)
        sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=AE_EPOCHS)
        best_loss, patience_cnt, best_state = 1e9, 0, None

        self.model.train()
        for epoch in range(1, AE_EPOCHS + 1):
            losses = []
            for (xb,) in loader:
                xb   = xb.to(DEVICE)
                loss = nn.functional.mse_loss(self.model(xb), xb)
                opt.zero_grad(); loss.backward(); opt.step()
                losses.append(loss.item())
            train_loss = float(np.mean(losses))
            sched.step()

            if train_loss < best_loss - 1e-6:
                best_loss, patience_cnt = train_loss, 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                patience_cnt += 1
                if patience_cnt >= AE_PATIENCE:
                    print(f"  AE early stop at epoch {epoch}  loss={train_loss:.5f}")
                    break
            if epoch % 20 == 0:
                print(f"  AE epoch {epoch:3d}/{AE_EPOCHS}  loss={train_loss:.5f}")

        self.model.load_state_dict(best_state)
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        Xt = torch.tensor(self.scaler.transform(X), dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            err = ((self.model(Xt) - Xt) ** 2).mean(dim=1).cpu().numpy()
        return err


print("Models defined.")

Models defined.


## 4. Train & Evaluate

Three changes vs the previous version:

1. **Threshold = 99th percentile of *training-normal* scores** — no abnormal labels needed, stable even when val contains very few anomalies. Normalisation also uses training score range so the threshold is consistent at inference time.
2. **Bandpass 300–4000 Hz** applied in feature extraction above.
3. **AE weight = 0.85, GMM = 0.15** — reflects the measured performance gap.

In [9]:
def norm_with_train(test_scores: np.ndarray, train_scores: np.ndarray) -> np.ndarray:
    """Normalise test scores using the training score range for consistent scaling."""
    lo, hi = train_scores.min(), train_scores.max()
    return (test_scores - lo) / (hi - lo + 1e-12)


results = []

for (mid, snr), grp in df.groupby(['machine_id', 'snr']):
    idx   = grp.index.values
    y_grp = y[idx]
    X_grp = X[idx]

    normal_idx   = np.where(y_grp == 0)[0]
    abnormal_idx = np.where(y_grp == 1)[0]

    print(f"\n{'='*55}")
    print(f"ID={mid}  SNR={snr}  normal={len(normal_idx)}  abnormal={len(abnormal_idx)}")

    if len(normal_idx) < 15 or len(abnormal_idx) < 2:
        print("  Skipping — not enough samples")
        continue

    # Split: 70% train-normal / 15% held-out-normal / 15% test-normal + all abnormal
    rng = np.random.default_rng(SEED)
    rng.shuffle(normal_idx)
    n_held  = max(2, int(len(normal_idx) * 0.15))
    n_test  = max(2, int(len(normal_idx) * 0.15))
    test_n  = normal_idx[n_held:n_held + n_test]
    train_n = normal_idx[n_held + n_test:]

    test_idx = np.concatenate([test_n, abnormal_idx])
    X_train  = X_grp[train_n]
    X_test   = X_grp[test_idx]
    y_test   = y_grp[test_idx]

    print(f"  train_normal={len(train_n)}  test={len(test_idx)}")

    # ── Fit ───────────────────────────────────────────────────────────────────
    ae_det  = AEDetector(FEAT_DIM).fit(X_train)
    gmm_det = GMMDetector().fit(X_train)

    # ── Training-normal scores → threshold (point 1) ──────────────────────────
    ae_train  = ae_det.score(X_train)
    gmm_train = gmm_det.score(X_train)

    # Fuse on training data using training-normalised scores
    fused_train = (W_AE  * norm_with_train(ae_train,  ae_train)
                 + W_GMM * norm_with_train(gmm_train, gmm_train))
    threshold = float(np.quantile(fused_train, THRESHOLD_QUANTILE))

    # ── Test scores (normalised with training range) ───────────────────────────
    ae_test   = norm_with_train(ae_det.score(X_test),  ae_train)
    gmm_test  = norm_with_train(gmm_det.score(X_test), gmm_train)
    fused_test = W_AE * ae_test + W_GMM * gmm_test

    # ── Metrics ───────────────────────────────────────────────────────────────
    ae_auc    = roc_auc_score(y_test, ae_test)
    gmm_auc   = roc_auc_score(y_test, gmm_test)
    fused_auc = roc_auc_score(y_test, fused_test)

    print(f"  AE AUC={ae_auc:.4f}   GMM AUC={gmm_auc:.4f}   Ensemble AUC={fused_auc:.4f}")
    print(f"  Threshold (q={THRESHOLD_QUANTILE}) = {threshold:.4f}")

    preds = (fused_test >= threshold).astype(int)
    print(classification_report(y_test, preds,
                                 target_names=['normal', 'abnormal'], zero_division=0))

    results.append(dict(
        machine_id=mid, snr=snr,
        ae_auc=ae_auc, gmm_auc=gmm_auc, auc=fused_auc,
        threshold=threshold,
        n_train=len(train_n), n_test=len(test_idx),
    ))

results_df = pd.DataFrame(results)
print("\n" + "="*55)
print(results_df.to_string(index=False))
print(f"\nMean ensemble AUC : {results_df.auc.mean():.4f}")
print(f"Mean AE AUC       : {results_df.ae_auc.mean():.4f}")
print(f"Mean GMM AUC      : {results_df.gmm_auc.mean():.4f}")


ID=00  SNR=0db  normal=1011  abnormal=407
  train_normal=709  test=558
  AE epoch  20/120  loss=0.30643
  AE epoch  40/120  loss=0.25320
  AE epoch  60/120  loss=0.28179
  AE epoch  80/120  loss=0.24087
  AE early stop at epoch 81  loss=0.25776
  GMM fitted  BIC=747829  components=8
  AE AUC=0.9202   GMM AUC=0.7352   Ensemble AUC=0.7476
  Threshold (q=0.99) = 0.6135
              precision    recall  f1-score   support

      normal       0.94      0.38      0.54       151
    abnormal       0.81      0.99      0.89       407

    accuracy                           0.83       558
   macro avg       0.87      0.69      0.72       558
weighted avg       0.85      0.83      0.80       558


ID=02  SNR=-6db  normal=1016  abnormal=359
  train_normal=712  test=511
  AE epoch  20/120  loss=0.27837
  AE epoch  40/120  loss=0.23710
  AE epoch  60/120  loss=0.21496
  AE epoch  80/120  loss=0.20396
  AE epoch 100/120  loss=0.22650
  AE early stop at epoch 114  loss=0.19179
  GMM fitted  BIC=7415

## 5. General-Use Detector

Train once on any machine's normal audio files, then call `predict_file()` on new recordings.
Works for any machine type — not just fans.

In [ ]:
class AnomalyDetector:
    """
    General-use unsupervised anomaly detector for machine audio.
    Train once on normal recordings, then score any new file.

    Usage:
        det = AnomalyDetector()
        det.fit(list_of_normal_wav_paths)
        print(det.predict_file("new_recording.wav"))
        # {'score': 0.73, 'threshold': 0.61, 'anomaly': True}
    """

    def __init__(self, threshold_quantile: float = THRESHOLD_QUANTILE):
        self.q              = threshold_quantile
        self.ae             = AEDetector(2 * N_MELS + 2 * N_MFCC + 4)
        self.gmm            = GMMDetector()
        self.threshold      = None
        self._ae_train_min  = None
        self._ae_train_max  = None
        self._gmm_train_min = None
        self._gmm_train_max = None

    def fit(self, normal_paths: list):
        """Train on a list of normal .wav paths."""
        print(f"Extracting features from {len(normal_paths)} normal files...")
        X_normal = np.stack([extract_features(p) for p in normal_paths])

        print("Training AE...")
        self.ae.fit(X_normal)
        print("Training GMM...")
        self.gmm.fit(X_normal)

        ae_s  = self.ae.score(X_normal)
        gmm_s = self.gmm.score(X_normal)

        # Store training score range for consistent inference-time normalisation
        self._ae_train_min,  self._ae_train_max  = ae_s.min(),  ae_s.max()
        self._gmm_train_min, self._gmm_train_max = gmm_s.min(), gmm_s.max()

        fused = self._fuse(ae_s, gmm_s)
        self.threshold = float(np.quantile(fused, self.q))
        print(f"Threshold set (q={self.q}) -> {self.threshold:.4f}")
        return self

    def _fuse(self, ae_s: np.ndarray, gmm_s: np.ndarray) -> np.ndarray:
        ae_n  = (ae_s  - self._ae_train_min)  / (self._ae_train_max  - self._ae_train_min  + 1e-12)
        gmm_n = (gmm_s - self._gmm_train_min) / (self._gmm_train_max - self._gmm_train_min + 1e-12)
        return W_AE * ae_n + W_GMM * gmm_n

    def predict_file(self, path: str) -> dict:
        """Score a single audio file."""
        if self.threshold is None:
            raise RuntimeError("Call .fit() before .predict_file()")
        feat  = extract_features(path).reshape(1, -1)
        score = float(self._fuse(self.ae.score(feat), self.gmm.score(feat))[0])
        return {'score': score, 'threshold': self.threshold, 'anomaly': score >= self.threshold}

    def predict_batch(self, paths: list) -> pd.DataFrame:
        """Score a list of files. Returns a DataFrame."""
        return pd.DataFrame([{'path': p, **self.predict_file(p)} for p in paths])

print("AnomalyDetector ready.")

AnomalyDetector ready.


In [11]:
normal_paths = df[(df.machine_id=='00') & (df.snr=='0db') & (df.label==0)]['path'].tolist()
test_paths   = df[(df.machine_id=='00') & (df.snr=='0db')]['path'].tolist()
test_labels  = df[(df.machine_id=='00') & (df.snr=='0db')]['label'].values

det = AnomalyDetector()
det.fit(normal_paths)
pred_df = det.predict_batch(test_paths)
pred_df['true_label'] = test_labels
print(pred_df.head(10))
print(f"AUC: {roc_auc_score(test_labels, pred_df.score):.4f}")

Extracting features from 1011 normal files...
Training AE...
  AE epoch  20/120  loss=0.25573
  AE epoch  40/120  loss=0.21040
  AE epoch  60/120  loss=0.19234
  AE epoch  80/120  loss=0.18141
  AE epoch 100/120  loss=0.17502
  AE epoch 120/120  loss=0.17200
Training GMM...
  GMM fitted  BIC=709161  components=8
Threshold set (q=0.99) -> 0.6253
                                                path     score  threshold  \
0  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  2.899882    0.62527   
1  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  2.575585    0.62527   
2  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  2.559187    0.62527   
3  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  1.295642    0.62527   
4  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  1.143688    0.62527   
5  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  2.795780    0.62527   
6  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DATASET\fan\id...  0.940540    0.62527   
7  D:\GITHUB\UBB-MGR-SEM3\PRZEMYSL\DA